# Medallion Architecture End-to-End Examples

This notebook walks through a simple bronze, silver, and gold pipeline in Databricks using Delta tables.

The flow covers:

- landing raw records in bronze
- cleaning and deduplicating into silver
- publishing a business aggregate in gold

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
bronze_table = "main.demo.orders_bronze"
silver_table = "main.demo.orders_silver"
gold_table = "main.demo.daily_sales_gold"

raw_orders = [
    (1001, "C001", "2026-04-01 08:10:00", "north", "120.50", "completed"),
    (1002, "C002", "2026-04-01 09:15:00", "south", "80.00", "completed"),
    (1002, "C002", "2026-04-01 09:20:00", "south", "82.00", "completed"),
    (1003, "C003", "2026-04-01 10:00:00", "west", "-10.00", "cancelled"),
    (1004, "C004", "bad_timestamp", "east", "50.00", "completed"),
    (None, "C005", "2026-04-01 11:45:00", "north", "44.00", "completed")
]

raw_df = spark.createDataFrame(
    raw_orders,
    ["order_id", "customer_id", "order_ts_raw", "region_raw", "amount_raw", "order_status_raw"]
)

## Bronze layer

Bronze keeps the source data close to the input shape and adds ingestion metadata.

In [ ]:
bronze_df = (
    raw_df
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_system", F.lit("orders_app"))
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)
print(f"Created or replaced {bronze_table}")

In [ ]:
display(spark.table(bronze_table).orderBy(F.col("order_id").asc_nulls_last(), "order_ts_raw"))

## Silver layer

Silver parses types, filters bad data, and keeps the latest valid record for each order.

In [ ]:
bronze_read_df = spark.table(bronze_table)

typed_df = (
    bronze_read_df
    .withColumn("order_ts", F.to_timestamp("order_ts_raw"))
    .withColumn("amount", F.col("amount_raw").cast("double"))
    .withColumn("region", F.upper(F.trim("region_raw")))
    .withColumn("order_status", F.lower(F.trim("order_status_raw")))
    .withColumn("order_date", F.to_date("order_ts"))
)

valid_df = typed_df.filter(
    F.col("order_id").isNotNull() &
    F.col("order_ts").isNotNull() &
    F.col("amount").isNotNull() &
    (F.col("amount") > 0)
)

latest_order_window = Window.partitionBy("order_id").orderBy(F.col("order_ts").desc())

silver_df = (
    valid_df
    .withColumn("row_num", F.row_number().over(latest_order_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .select(
        "order_id",
        "customer_id",
        "order_ts",
        "order_date",
        "region",
        "amount",
        "order_status",
        "ingest_ts",
        "source_system"
    )
)

silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
print(f"Created or replaced {silver_table}")

In [ ]:
display(spark.table(silver_table).orderBy("order_id"))

In [ ]:
invalid_record_count = typed_df.count() - silver_df.count()
print(f"Records rejected before silver publishing: {invalid_record_count}")

## Gold layer

Gold serves a business question. Here the output is daily completed sales by region.

In [ ]:
gold_df = (
    spark.table(silver_table)
    .filter(F.col("order_status") == "completed")
    .groupBy("order_date", "region")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("amount").alias("gross_sales")
    )
    .orderBy("order_date", "region")
)

gold_df.write.format("delta").mode("overwrite").saveAsTable(gold_table)
print(f"Created or replaced {gold_table}")

In [ ]:
display(spark.table(gold_table))

## Production notes

- Bronze should preserve raw fidelity and ingestion metadata.
- Silver is the contract layer for cleaned and reusable data.
- Gold should stay aligned to a specific business output.
- In production, replace overwrite patterns with incremental merges and orchestration through Databricks Jobs.